In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Check input data and model files

In [2]:
from pathlib import Path
import json
import joblib
import pandas as pd

ARTIFACT_DIR = Path(
    "/kaggle/input/datasets/huangyufeid/cite-final-artifacts"
)

MODEL_PATH = (
    ARTIFACT_DIR
    / "cite_svd128_celltype_ridge1000.joblib"
)

PREDICTION_PATH = (
    ARTIFACT_DIR
    / "cite_test_predictions_svd128_celltype_ridge1000.h5"
)

CONFIG_PATH = (
    ARTIFACT_DIR
    / "model_config.json"
)

required_files = [
    MODEL_PATH,
    PREDICTION_PATH,
    CONFIG_PATH,
]

print("Artifact check")
print("-" * 70)

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    size_mb = path.stat().st_size / 1024**2 if path.is_file() else 0

    print(
        f"{path.name:<55} "
        f"{status:<8} "
        f"{size_mb:.2f} MB"
    )

if not all(path.is_file() for path in required_files):
    raise FileNotFoundError(
        "部分 CITE artifacts 没有找到，请检查 Dataset 路径。"
    )

Artifact check
----------------------------------------------------------------------
cite_svd128_celltype_ridge1000.joblib                   OK       10.37 MB
cite_test_predictions_svd128_celltype_ridge1000.h5      OK       21.81 MB
model_config.json                                       OK       0.00 MB


In [3]:
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    model_config = json.load(file)

print(
    json.dumps(
        model_config,
        indent=2,
        ensure_ascii=False
    )
)

{
  "model_name": "CITE SVD128 + cell type + Ridge1000",
  "latent_dim": 128,
  "ridge_alpha": 1000.0,
  "random_state": 42,
  "use_day": false,
  "use_donor": false,
  "use_cell_type": true,
  "n_train_cells": 70988,
  "n_genes": 22050,
  "n_proteins": 140,
  "n_final_features": 135,
  "explained_variance": 0.13582591712474823,
  "train_score": 0.891259491443634,
  "numpy_version": "2.0.2",
  "pandas_version": "2.3.3",
  "sklearn_version": "1.6.1",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]"
}


In [4]:
cite_model = joblib.load(MODEL_PATH)

print("Model object type:")
print(type(cite_model))

if isinstance(cite_model, dict):
    print("\nDictionary keys:")
    for key in cite_model.keys():
        value = cite_model[key]
        print(f"{key}: {type(value)}")

elif hasattr(cite_model, "named_steps"):
    print("\nPipeline steps:")
    for name, component in cite_model.named_steps.items():
        print(f"{name}: {type(component)}")

else:
    print("\nModel attributes:")
    possible_attributes = [
        "n_components",
        "alpha",
        "coef_",
        "intercept_",
        "feature_names_in_",
        "classes_",
    ]

    for attribute in possible_attributes:
        if hasattr(cite_model, attribute):
            value = getattr(cite_model, attribute)

            if hasattr(value, "shape"):
                print(attribute, "shape =", value.shape)
            else:
                print(attribute, "=", value)

Model object type:
<class 'dict'>

Dictionary keys:
config: <class 'dict'>
svd: <class 'sklearn.decomposition._truncated_svd.TruncatedSVD'>
rna_scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
cell_type_encoder: <class 'sklearn.preprocessing._encoders.OneHotEncoder'>
cell_type_scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
ridge: <class 'sklearn.linear_model._ridge.Ridge'>
gene_ids: <class 'numpy.ndarray'>
protein_ids: <class 'numpy.ndarray'>
cell_type_feature_names: <class 'numpy.ndarray'>


In [5]:
with pd.HDFStore(PREDICTION_PATH, mode="r") as store:
    prediction_keys = store.keys()

print("HDF5 keys:", prediction_keys)

HDF5 keys: ['/predictions']


In [6]:
cite_test_predictions = pd.read_hdf(
    PREDICTION_PATH
)

print("Prediction type:", type(cite_test_predictions))
print("Prediction shape:", cite_test_predictions.shape)
print("\nFirst five rows:")
display(cite_test_predictions.head())

print("\nIndex name:", cite_test_predictions.index.name)
print("Columns:", cite_test_predictions.columns[:10].tolist())

Prediction type: <class 'pandas.core.frame.DataFrame'>
Prediction shape: (48663, 140)

First five rows:


gene_id,CD86,CD274,CD270,CD155,CD112,CD47,CD48,CD40,CD154,CD52,...,CD94,CD162,CD85j,CD23,CD328,HLA-E,CD82,CD101,CD88,CD224
cell_id,,,,,,,,,,,,,,,,,,,,,
c2150f55becb,0.537193,0.402024,0.827317,4.379986,5.268987,7.627712,4.852458,0.435169,0.553981,0.926758,...,0.330406,5.384054,0.416561,0.415530,0.424005,0.720599,3.424686,0.754289,1.427210,2.140374
65b7edf8a4da,0.460908,0.316873,0.524985,3.762704,4.861040,5.152702,2.883190,0.157213,0.380975,0.330094,...,0.302149,5.035058,0.243428,0.402201,0.356731,0.425718,2.126056,0.448299,1.401048,1.091228
c1b26cb1057b,0.200529,0.289051,0.695400,1.548912,0.499208,4.321531,0.082610,0.041116,0.800055,-0.094935,...,0.013922,6.639294,1.402478,-0.170013,0.147496,0.537220,8.625221,0.273981,6.566155,2.527836
917168fa6f83,0.553623,0.215023,0.566204,6.390348,6.597325,6.261046,12.489809,0.291535,0.385098,0.061624,...,0.206050,10.813756,0.378859,0.180043,0.011671,0.496738,3.804829,0.558055,1.922254,3.274741
2b29feeca86d,0.216965,0.325564,0.921159,3.639483,1.929940,3.617947,0.261051,0.134104,0.929498,-0.211988,...,0.110576,7.056181,1.576681,-0.173772,0.439444,0.732424,5.111184,0.309405,7.613845,2.210604



Index name: cell_id
Columns: ['CD86', 'CD274', 'CD270', 'CD155', 'CD112', 'CD47', 'CD48', 'CD40', 'CD154', 'CD52']


In [7]:
import numpy as np

prediction_array = cite_test_predictions.to_numpy()

print("Contains NaN:", np.isnan(prediction_array).any())
print("Contains inf:", np.isinf(prediction_array).any())
print("Minimum:", prediction_array.min())
print("Maximum:", prediction_array.max())
print("Mean:", prediction_array.mean())
print("Standard deviation:", prediction_array.std())

assert prediction_array.shape[1] == 140
assert np.isfinite(prediction_array).all()

Contains NaN: False
Contains inf: False
Minimum: -5.158785
Maximum: 48.47808
Mean: 2.0374892
Standard deviation: 3.3823118


In [8]:
print("Saved model component parameters")
print("=" * 70)

for component_name in [
    "svd",
    "rna_scaler",
    "cell_type_encoder",
    "cell_type_scaler",
    "ridge",
]:
    component = cite_model[component_name]

    print(f"\n[{component_name}]")
    print(component.get_params())

Saved model component parameters

[svd]
{'algorithm': 'randomized', 'n_components': 128, 'n_iter': 5, 'n_oversamples': 10, 'power_iteration_normalizer': 'auto', 'random_state': 42, 'tol': 0.0}

[rna_scaler]
{'copy': True, 'with_mean': True, 'with_std': True}

[cell_type_encoder]
{'categories': 'auto', 'drop': None, 'dtype': <class 'numpy.float32'>, 'feature_name_combiner': 'concat', 'handle_unknown': 'ignore', 'max_categories': None, 'min_frequency': None, 'sparse_output': False}

[cell_type_scaler]
{'copy': True, 'with_mean': True, 'with_std': True}

[ridge]
{'alpha': 1000.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [9]:
from pathlib import Path

possible_roots = [
    Path("/kaggle/input/competitions/open-problems-multimodal"),
    Path("/kaggle/input/open-problems-multimodal"),
]

COMPETITION_DIR = None

for root in possible_roots:
    if (root / "train_cite_inputs.h5").is_file():
        COMPETITION_DIR = root
        break

if COMPETITION_DIR is None:
    candidates = list(
        Path("/kaggle/input").rglob("train_cite_inputs.h5")
    )

    if len(candidates) == 0:
        raise FileNotFoundError(
            "没有找到 train_cite_inputs.h5。"
            "请确认已经通过 Add Input 添加 competition dataset。"
        )

    COMPETITION_DIR = candidates[0].parent

print("Competition directory:")
print(COMPETITION_DIR)

required_training_files = [
    COMPETITION_DIR / "train_cite_inputs.h5",
    COMPETITION_DIR / "train_cite_targets.h5",
    COMPETITION_DIR / "metadata.csv",
]

for path in required_training_files:
    print(path.name, "OK" if path.is_file() else "MISSING")

Competition directory:
/kaggle/input/competitions/open-problems-multimodal
train_cite_inputs.h5 OK
train_cite_targets.h5 OK
metadata.csv OK


In [10]:
import numpy as np
import pandas as pd

print("Loading CITE input matrix...")

X_cite = pd.read_hdf(
    COMPETITION_DIR / "train_cite_inputs.h5"
)

print("Loading CITE target matrix...")

Y_cite = pd.read_hdf(
    COMPETITION_DIR / "train_cite_targets.h5"
)

print("Loading metadata...")

metadata = pd.read_csv(
    COMPETITION_DIR / "metadata.csv",
    index_col="cell_id"
)

# 按 CITE cell ID 对齐
meta_cite = metadata.loc[X_cite.index].copy()

print("\nShapes")
print("-" * 50)
print("X_cite:", X_cite.shape)
print("Y_cite:", Y_cite.shape)
print("meta_cite:", meta_cite.shape)

assert X_cite.index.equals(Y_cite.index)
assert X_cite.index.equals(meta_cite.index)

# 检查 feature 顺序与最终模型记录是否一致
assert np.array_equal(
    X_cite.columns.to_numpy(),
    cite_model["gene_ids"]
), "RNA gene order does not match the saved model."

assert np.array_equal(
    Y_cite.columns.to_numpy(),
    cite_model["protein_ids"]
), "Protein order does not match the saved model."

print("\nGene and protein orders match the saved model.")

print("\nDay distribution:")
print(meta_cite["day"].value_counts().sort_index())

print("\nDonor distribution:")
print(meta_cite["donor"].value_counts().sort_index())

print("\nCell-type distribution:")
print(meta_cite["cell_type"].value_counts())

Loading CITE input matrix...
Loading CITE target matrix...
Loading metadata...

Shapes
--------------------------------------------------
X_cite: (70988, 22050)
Y_cite: (70988, 140)
meta_cite: (70988, 4)

Gene and protein orders match the saved model.

Day distribution:
day
2    21942
3    20901
4    28145
Name: count, dtype: int64

Donor distribution:
donor
13176    22199
31800    24803
32606    23986
Name: count, dtype: int64

Cell-type distribution:
cell_type
HSC     29879
EryP    14241
NeuP    12493
MasP     8242
MkP      5382
MoP       591
BP        160
Name: count, dtype: int64


In [11]:
def rowwise_pearson_score(
    y_true,
    y_pred,
    return_per_cell=False
):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"Shape mismatch: "
            f"y_true={y_true.shape}, "
            f"y_pred={y_pred.shape}"
        )

    y_true_centered = (
        y_true
        - y_true.mean(axis=1, keepdims=True)
    )

    y_pred_centered = (
        y_pred
        - y_pred.mean(axis=1, keepdims=True)
    )

    numerator = np.sum(
        y_true_centered * y_pred_centered,
        axis=1
    )

    denominator = np.sqrt(
        np.sum(y_true_centered ** 2, axis=1)
        *
        np.sum(y_pred_centered ** 2, axis=1)
    )

    cell_correlations = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 0
    )

    if return_per_cell:
        return cell_correlations

    return float(cell_correlations.mean())


# Sanity check
perfect_score = rowwise_pearson_score(
    Y_cite.iloc[:1000],
    Y_cite.iloc[:1000]
)

print("Perfect prediction score:", perfect_score)

assert np.isclose(perfect_score, 1.0)

Perfect prediction score: 1.0


In [12]:
HELD_OUT_DONOR = "32606"
TRAIN_DAYS = [2, 3]
VALID_DAY = 4

donor_string = meta_cite["donor"].astype(str)

train_mask = (
    meta_cite["day"].isin(TRAIN_DAYS)
    &
    donor_string.ne(HELD_OUT_DONOR)
)

valid_mask = (
    meta_cite["day"].eq(VALID_DAY)
    &
    donor_string.eq(HELD_OUT_DONOR)
)

train_indices = np.flatnonzero(
    train_mask.to_numpy()
)

valid_indices = np.flatnonzero(
    valid_mask.to_numpy()
)

print("Strict validation split")
print("=" * 60)
print("Held-out donor:", HELD_OUT_DONOR)
print("Training days:", TRAIN_DAYS)
print("Validation day:", VALID_DAY)
print("Training cells:", len(train_indices))
print("Validation cells:", len(valid_indices))

assert len(train_indices) > 0
assert len(valid_indices) > 0

assert np.intersect1d(
    train_indices,
    valid_indices
).size == 0

print("\nTraining donor × day:")
display(
    meta_cite.iloc[train_indices]
    .groupby(["donor", "day"])
    .size()
    .unstack(fill_value=0)
)

print("\nValidation donor × day:")
display(
    meta_cite.iloc[valid_indices]
    .groupby(["donor", "day"])
    .size()
    .unstack(fill_value=0)
)

print("\nTraining cell types:")
print(
    meta_cite.iloc[train_indices]["cell_type"]
    .value_counts()
)

print("\nValidation cell types:")
print(
    meta_cite.iloc[valid_indices]["cell_type"]
    .value_counts()
)

Strict validation split
Held-out donor: 32606
Training days: [2, 3]
Validation day: 4
Training cells: 28368
Validation cells: 9511

Training donor × day:


day,2,3
donor,,
13176,6071,7643
31800,8395,6259



Validation donor × day:


day,4
donor,
32606,9511



Training cell types:
cell_type
HSC     13653
EryP     4915
NeuP     4557
MkP      2659
MasP     2305
MoP       187
BP         92
Name: count, dtype: int64

Validation cell types:
cell_type
HSC     3131
EryP    2198
NeuP    1935
MasP    1622
MkP      541
MoP       73
BP        11
Name: count, dtype: int64


In [13]:
Y_train = Y_cite.iloc[
    train_indices
].to_numpy(dtype=np.float32)

Y_valid = Y_cite.iloc[
    valid_indices
].to_numpy(dtype=np.float32)

training_mean_profile = Y_train.mean(axis=0)

Y_pred_mean = np.broadcast_to(
    training_mean_profile,
    Y_valid.shape
)

strict_mean_score = rowwise_pearson_score(
    Y_valid,
    Y_pred_mean
)

print("Strict mean-profile baseline:", strict_mean_score)

Strict mean-profile baseline: 0.7761645130546844


In [16]:
import time
from sklearn.base import clone

fold_svd = clone(cite_model["svd"])

print("Fold SVD parameters:")
print(fold_svd.get_params())

X_train_raw = X_cite.iloc[train_indices]
X_valid_raw = X_cite.iloc[valid_indices]

start_time = time.time()

print("\nFitting fold-specific SVD...")

X_train_svd = fold_svd.fit_transform(
    X_train_raw
).astype(np.float32)

X_valid_svd = fold_svd.transform(
    X_valid_raw
).astype(np.float32)

svd_minutes = (time.time() - start_time) / 60

print("X_train_svd shape:", X_train_svd.shape)
print("X_valid_svd shape:", X_valid_svd.shape)
print(
    "Explained variance:",
    float(fold_svd.explained_variance_ratio_.sum())
)
print("SVD time:", svd_minutes, "minutes")

Fold SVD parameters:
{'algorithm': 'randomized', 'n_components': 128, 'n_iter': 5, 'n_oversamples': 10, 'power_iteration_normalizer': 'auto', 'random_state': 42, 'tol': 0.0}

Fitting fold-specific SVD...
X_train_svd shape: (28368, 128)
X_valid_svd shape: (9511, 128)
Explained variance: 0.14364372193813324
SVD time: 0.43546148935953777 minutes


In [17]:
fold_rna_scaler = clone(
    cite_model["rna_scaler"]
)

X_train_rna_scaled = (
    fold_rna_scaler
    .fit_transform(X_train_svd)
    .astype(np.float32)
)

X_valid_rna_scaled = (
    fold_rna_scaler
    .transform(X_valid_svd)
    .astype(np.float32)
)

print(
    "Scaled RNA train:",
    X_train_rna_scaled.shape
)

print(
    "Scaled RNA valid:",
    X_valid_rna_scaled.shape
)

Scaled RNA train: (28368, 128)
Scaled RNA valid: (9511, 128)


In [18]:
from scipy import sparse

fold_cell_encoder = clone(
    cite_model["cell_type_encoder"]
)

fold_cell_scaler = clone(
    cite_model["cell_type_scaler"]
)

cell_train = (
    meta_cite.iloc[train_indices][["cell_type"]]
)

cell_valid = (
    meta_cite.iloc[valid_indices][["cell_type"]]
)

cell_train_encoded = (
    fold_cell_encoder.fit_transform(cell_train)
)

cell_valid_encoded = (
    fold_cell_encoder.transform(cell_valid)
)

# cell type 只有少数几列，转为 dense 很安全
if sparse.issparse(cell_train_encoded):
    cell_train_encoded = (
        cell_train_encoded.toarray()
    )

if sparse.issparse(cell_valid_encoded):
    cell_valid_encoded = (
        cell_valid_encoded.toarray()
    )

cell_train_encoded = np.asarray(
    cell_train_encoded,
    dtype=np.float32
)

cell_valid_encoded = np.asarray(
    cell_valid_encoded,
    dtype=np.float32
)

cell_train_scaled = (
    fold_cell_scaler
    .fit_transform(cell_train_encoded)
    .astype(np.float32)
)

cell_valid_scaled = (
    fold_cell_scaler
    .transform(cell_valid_encoded)
    .astype(np.float32)
)

print(
    "Cell-type categories:",
    fold_cell_encoder.categories_[0]
)

print(
    "Cell-type train features:",
    cell_train_scaled.shape
)

print(
    "Cell-type valid features:",
    cell_valid_scaled.shape
)

Cell-type categories: ['BP' 'EryP' 'HSC' 'MasP' 'MkP' 'MoP' 'NeuP']
Cell-type train features: (28368, 7)
Cell-type valid features: (9511, 7)


In [19]:
X_train_final = np.hstack(
    [
        X_train_rna_scaled,
        cell_train_scaled,
    ]
).astype(np.float32)

X_valid_final = np.hstack(
    [
        X_valid_rna_scaled,
        cell_valid_scaled,
    ]
).astype(np.float32)

print("Final train features:", X_train_final.shape)
print("Final valid features:", X_valid_final.shape)

assert X_train_final.shape[0] == len(train_indices)
assert X_valid_final.shape[0] == len(valid_indices)

Final train features: (28368, 135)
Final valid features: (9511, 135)


In [20]:
fold_ridge = clone(
    cite_model["ridge"]
)

print("Ridge parameters:")
print(fold_ridge.get_params())

start_time = time.time()

fold_ridge.fit(
    X_train_final,
    Y_train
)

Y_pred_ridge = fold_ridge.predict(
    X_valid_final
).astype(np.float32)

ridge_minutes = (time.time() - start_time) / 60

strict_ridge_score = rowwise_pearson_score(
    Y_valid,
    Y_pred_ridge
)

print("\nStrict validation results")
print("=" * 60)
print(
    "Mean-profile baseline:",
    strict_mean_score
)
print(
    "SVD128 + cell type + Ridge1000:",
    strict_ridge_score
)
print(
    "Absolute improvement:",
    strict_ridge_score - strict_mean_score
)
print(
    "Relative error reduction:",
    (
        (strict_ridge_score - strict_mean_score)
        / (1.0 - strict_mean_score)
    )
)
print("Ridge time:", ridge_minutes, "minutes")

Ridge parameters:
{'alpha': 1000.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}

Strict validation results
Mean-profile baseline: 0.7761645130546844
SVD128 + cell type + Ridge1000: 0.8791023592630038
Absolute improvement: 0.10293784620831947
Relative error reduction: 0.45988170871881373
Ridge time: 0.0015395124753316244 minutes


In [21]:
cell_correlations = rowwise_pearson_score(
    Y_valid,
    Y_pred_ridge,
    return_per_cell=True
)

strict_cell_results = (
    meta_cite.iloc[valid_indices][
        ["day", "donor", "cell_type"]
    ]
    .copy()
)

strict_cell_results["cell_correlation"] = (
    cell_correlations
)

cell_type_summary = (
    strict_cell_results
    .groupby("cell_type")
    .agg(
        n_cells=("cell_correlation", "size"),
        mean_correlation=(
            "cell_correlation",
            "mean"
        ),
        std_correlation=(
            "cell_correlation",
            "std"
        ),
        median_correlation=(
            "cell_correlation",
            "median"
        ),
    )
    .sort_values(
        "mean_correlation",
        ascending=False
    )
)

display(cell_type_summary)

,n_cells,mean_correlation,std_correlation,median_correlation
cell_type,,,,
HSC,3131,0.902448,0.054541,0.912113
NeuP,1935,0.888067,0.058105,0.898659
MasP,1622,0.866831,0.064848,0.876607
MkP,541,0.857781,0.050089,0.865584
EryP,2198,0.853437,0.066511,0.865954
BP,11,0.849652,0.033740,0.853322
MoP,73,0.848061,0.068342,0.861330


In [22]:
from pathlib import Path

summary_result = pd.DataFrame(
    [
        {
            "task": "CITE",
            "validation": "strict_donor_time",
            "held_out_donor": HELD_OUT_DONOR,
            "train_days": ",".join(
                map(str, TRAIN_DAYS)
            ),
            "valid_day": VALID_DAY,
            "n_train": len(train_indices),
            "n_valid": len(valid_indices),
            "mean_baseline": strict_mean_score,
            "model_score": strict_ridge_score,
            "absolute_improvement": (
                strict_ridge_score
                - strict_mean_score
            ),
            "svd_components": (
                fold_svd.n_components
            ),
            "explained_variance": float(
                fold_svd
                .explained_variance_ratio_
                .sum()
            ),
            "ridge_alpha": fold_ridge.alpha,
            "use_cell_type": True,
        }
    ]
)

SUMMARY_PATH = Path(
    "/kaggle/working/"
    "cite_strict_validation_summary.csv"
)

CELL_RESULT_PATH = Path(
    "/kaggle/working/"
    "cite_strict_validation_cell_scores.csv"
)

CELL_TYPE_PATH = Path(
    "/kaggle/working/"
    "cite_strict_validation_celltype_summary.csv"
)

summary_result.to_csv(
    SUMMARY_PATH,
    index=False
)

strict_cell_results.to_csv(
    CELL_RESULT_PATH,
    index=True
)

cell_type_summary.to_csv(
    CELL_TYPE_PATH,
    index=True
)

display(summary_result)

print("\nSaved:")
print(SUMMARY_PATH)
print(CELL_RESULT_PATH)
print(CELL_TYPE_PATH)

,task,validation,held_out_donor,train_days,valid_day,n_train,n_valid,mean_baseline,model_score,absolute_improvement,svd_components,explained_variance,ridge_alpha,use_cell_type
0,CITE,strict_donor_time,32606,"2,3",4,28368,9511,0.776165,0.879102,0.102938,128,0.143644,1000.0,True



Saved:
/kaggle/working/cite_strict_validation_summary.csv
/kaggle/working/cite_strict_validation_cell_scores.csv
/kaggle/working/cite_strict_validation_celltype_summary.csv


RNA-only

In [23]:
from sklearn.base import clone

rna_only_ridge = clone(
    cite_model["ridge"]
)

rna_only_ridge.fit(
    X_train_rna_scaled,
    Y_train
)

Y_pred_rna_only = (
    rna_only_ridge
    .predict(X_valid_rna_scaled)
    .astype(np.float32)
)

strict_rna_only_score = rowwise_pearson_score(
    Y_valid,
    Y_pred_rna_only
)

cell_type_gain = (
    strict_ridge_score
    - strict_rna_only_score
)

print("Held-out donor:", HELD_OUT_DONOR)
print("Mean-profile baseline:", strict_mean_score)
print("RNA-only Ridge:", strict_rna_only_score)
print("RNA + cell type Ridge:", strict_ridge_score)
print("Cell-type gain:", cell_type_gain)

Held-out donor: 32606
Mean-profile baseline: 0.7761645130546844
RNA-only Ridge: 0.8789436932650831
RNA + cell type Ridge: 0.8791023592630038
Cell-type gain: 0.0001586659979206928


In [24]:
summary_result["rna_only_score"] = (
    strict_rna_only_score
)

summary_result["rna_celltype_score"] = (
    strict_ridge_score
)

summary_result["celltype_gain"] = (
    cell_type_gain
)

current_fold_path = Path(
    "/kaggle/working/"
    f"cite_strict_fold_{HELD_OUT_DONOR}_summary.csv"
)

summary_result.to_csv(
    current_fold_path,
    index=False
)

display(summary_result)

print("Saved:", current_fold_path)

,task,validation,held_out_donor,train_days,valid_day,n_train,n_valid,mean_baseline,model_score,absolute_improvement,svd_components,explained_variance,ridge_alpha,use_cell_type,rna_only_score,rna_celltype_score,celltype_gain
0,CITE,strict_donor_time,32606,"2,3",4,28368,9511,0.776165,0.879102,0.102938,128,0.143644,1000.0,True,0.878944,0.879102,0.000159


Saved: /kaggle/working/cite_strict_fold_32606_summary.csv


In [25]:
import gc
import time
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import sparse
from sklearn.base import clone


STRICT_OUTPUT_DIR = Path(
    "/kaggle/working/cite_strict_results"
)

STRICT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def run_cite_strict_fold(
    held_out_donor,
    train_days=(2, 3),
    valid_day=4
):
    held_out_donor = str(held_out_donor)

    print("\n" + "=" * 70)
    print("Strict CITE validation")
    print("Held-out donor:", held_out_donor)
    print("Training days:", train_days)
    print("Validation day:", valid_day)
    print("=" * 70)

    donor_string = (
        meta_cite["donor"].astype(str)
    )

    train_mask = (
        meta_cite["day"].isin(train_days)
        &
        donor_string.ne(held_out_donor)
    )

    valid_mask = (
        meta_cite["day"].eq(valid_day)
        &
        donor_string.eq(held_out_donor)
    )

    train_indices_fold = np.flatnonzero(
        train_mask.to_numpy()
    )

    valid_indices_fold = np.flatnonzero(
        valid_mask.to_numpy()
    )

    n_train = len(train_indices_fold)
    n_valid = len(valid_indices_fold)

    print("Training cells:", n_train)
    print("Validation cells:", n_valid)

    if n_train == 0 or n_valid == 0:
        raise ValueError(
            f"Empty split for donor "
            f"{held_out_donor}."
        )

    # -----------------------------------------------------
    # Targets and mean-profile baseline
    # -----------------------------------------------------

    Y_train_fold = (
        Y_cite.iloc[train_indices_fold]
        .to_numpy(dtype=np.float32)
    )

    Y_valid_fold = (
        Y_cite.iloc[valid_indices_fold]
        .to_numpy(dtype=np.float32)
    )

    train_mean_profile = (
        Y_train_fold.mean(axis=0)
    )

    Y_pred_mean_fold = np.broadcast_to(
        train_mean_profile,
        Y_valid_fold.shape
    )

    mean_score = rowwise_pearson_score(
        Y_valid_fold,
        Y_pred_mean_fold
    )

    print("Mean baseline:", mean_score)

    # -----------------------------------------------------
    # Fold-specific SVD
    # -----------------------------------------------------

    fold_svd = clone(
        cite_model["svd"]
    )

    start_time = time.time()

    print("Fitting fold-specific SVD...")

    X_train_svd_fold = (
        fold_svd.fit_transform(
            X_cite.iloc[train_indices_fold]
        )
        .astype(np.float32)
    )

    X_valid_svd_fold = (
        fold_svd.transform(
            X_cite.iloc[valid_indices_fold]
        )
        .astype(np.float32)
    )

    svd_time_minutes = (
        time.time() - start_time
    ) / 60

    explained_variance = float(
        fold_svd
        .explained_variance_ratio_
        .sum()
    )

    print(
        "Explained variance:",
        explained_variance
    )

    print(
        "SVD time:",
        svd_time_minutes,
        "minutes"
    )

    # -----------------------------------------------------
    # RNA scaling
    # -----------------------------------------------------

    fold_rna_scaler = clone(
        cite_model["rna_scaler"]
    )

    X_train_rna_fold = (
        fold_rna_scaler
        .fit_transform(X_train_svd_fold)
        .astype(np.float32)
    )

    X_valid_rna_fold = (
        fold_rna_scaler
        .transform(X_valid_svd_fold)
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # RNA-only Ridge
    # -----------------------------------------------------

    rna_only_ridge_fold = clone(
        cite_model["ridge"]
    )

    rna_only_ridge_fold.fit(
        X_train_rna_fold,
        Y_train_fold
    )

    Y_pred_rna_only_fold = (
        rna_only_ridge_fold
        .predict(X_valid_rna_fold)
        .astype(np.float32)
    )

    rna_only_score = rowwise_pearson_score(
        Y_valid_fold,
        Y_pred_rna_only_fold
    )

    print(
        "RNA-only score:",
        rna_only_score
    )

    # -----------------------------------------------------
    # Cell-type encoding
    # -----------------------------------------------------

    fold_cell_encoder = clone(
        cite_model["cell_type_encoder"]
    )

    fold_cell_scaler = clone(
        cite_model["cell_type_scaler"]
    )

    cell_train_fold = (
        meta_cite
        .iloc[train_indices_fold][
            ["cell_type"]
        ]
    )

    cell_valid_fold = (
        meta_cite
        .iloc[valid_indices_fold][
            ["cell_type"]
        ]
    )

    cell_train_encoded = (
        fold_cell_encoder
        .fit_transform(cell_train_fold)
    )

    cell_valid_encoded = (
        fold_cell_encoder
        .transform(cell_valid_fold)
    )

    if sparse.issparse(
        cell_train_encoded
    ):
        cell_train_encoded = (
            cell_train_encoded.toarray()
        )

    if sparse.issparse(
        cell_valid_encoded
    ):
        cell_valid_encoded = (
            cell_valid_encoded.toarray()
        )

    cell_train_encoded = np.asarray(
        cell_train_encoded,
        dtype=np.float32
    )

    cell_valid_encoded = np.asarray(
        cell_valid_encoded,
        dtype=np.float32
    )

    cell_train_scaled_fold = (
        fold_cell_scaler
        .fit_transform(cell_train_encoded)
        .astype(np.float32)
    )

    cell_valid_scaled_fold = (
        fold_cell_scaler
        .transform(cell_valid_encoded)
        .astype(np.float32)
    )

    # -----------------------------------------------------
    # RNA + cell-type features
    # -----------------------------------------------------

    X_train_final_fold = np.hstack(
        [
            X_train_rna_fold,
            cell_train_scaled_fold,
        ]
    ).astype(np.float32)

    X_valid_final_fold = np.hstack(
        [
            X_valid_rna_fold,
            cell_valid_scaled_fold,
        ]
    ).astype(np.float32)

    print(
        "Final feature dimension:",
        X_train_final_fold.shape[1]
    )

    # -----------------------------------------------------
    # Final Ridge
    # -----------------------------------------------------

    final_ridge_fold = clone(
        cite_model["ridge"]
    )

    final_ridge_fold.fit(
        X_train_final_fold,
        Y_train_fold
    )

    Y_pred_final_fold = (
        final_ridge_fold
        .predict(X_valid_final_fold)
        .astype(np.float32)
    )

    final_score = rowwise_pearson_score(
        Y_valid_fold,
        Y_pred_final_fold
    )

    celltype_gain = (
        final_score - rna_only_score
    )

    print(
        "RNA + cell-type score:",
        final_score
    )

    print(
        "Cell-type gain:",
        celltype_gain
    )

    # -----------------------------------------------------
    # Per-cell results
    # -----------------------------------------------------

    cell_scores = rowwise_pearson_score(
        Y_valid_fold,
        Y_pred_final_fold,
        return_per_cell=True
    )

    cell_result = (
        meta_cite
        .iloc[valid_indices_fold][
            ["day", "donor", "cell_type"]
        ]
        .copy()
    )

    cell_result[
        "cell_correlation"
    ] = cell_scores

    cell_result[
        "held_out_donor"
    ] = held_out_donor

    # -----------------------------------------------------
    # Fold summary
    # -----------------------------------------------------

    fold_summary = pd.DataFrame(
        [
            {
                "task": "CITE",
                "validation": (
                    "strict_donor_time"
                ),
                "held_out_donor": (
                    held_out_donor
                ),
                "train_days": ",".join(
                    map(str, train_days)
                ),
                "valid_day": valid_day,
                "n_train": n_train,
                "n_valid": n_valid,
                "mean_baseline": mean_score,
                "rna_only_score": (
                    rna_only_score
                ),
                "rna_celltype_score": (
                    final_score
                ),
                "celltype_gain": (
                    celltype_gain
                ),
                "improvement_over_mean": (
                    final_score - mean_score
                ),
                "svd_components": (
                    fold_svd.n_components
                ),
                "explained_variance": (
                    explained_variance
                ),
                "ridge_alpha": (
                    final_ridge_fold.alpha
                ),
                "svd_time_minutes": (
                    svd_time_minutes
                ),
            }
        ]
    )

    # -----------------------------------------------------
    # Save immediately
    # -----------------------------------------------------

    summary_path = (
        STRICT_OUTPUT_DIR
        / (
            f"cite_strict_fold_"
            f"{held_out_donor}_summary.csv"
        )
    )

    cell_path = (
        STRICT_OUTPUT_DIR
        / (
            f"cite_strict_fold_"
            f"{held_out_donor}_cell_scores.csv"
        )
    )

    fold_summary.to_csv(
        summary_path,
        index=False
    )

    cell_result.to_csv(
        cell_path,
        index=True
    )

    print("Saved:", summary_path)
    print("Saved:", cell_path)

    # -----------------------------------------------------
    # Memory cleanup
    # -----------------------------------------------------

    del (
        Y_train_fold,
        Y_valid_fold,
        Y_pred_mean_fold,
        X_train_svd_fold,
        X_valid_svd_fold,
        X_train_rna_fold,
        X_valid_rna_fold,
        cell_train_encoded,
        cell_valid_encoded,
        cell_train_scaled_fold,
        cell_valid_scaled_fold,
        X_train_final_fold,
        X_valid_final_fold,
        Y_pred_rna_only_fold,
        Y_pred_final_fold,
    )

    gc.collect()

    return fold_summary, cell_result

other 2 doners

In [26]:
remaining_donors = [
    "13176",
    "31800",
]

new_fold_summaries = []
new_cell_results = []

for donor in remaining_donors:
    fold_summary, fold_cell_result = (
        run_cite_strict_fold(
            held_out_donor=donor,
            train_days=(2, 3),
            valid_day=4
        )
    )

    new_fold_summaries.append(
        fold_summary
    )

    new_cell_results.append(
        fold_cell_result
    )


Strict CITE validation
Held-out donor: 13176
Training days: (2, 3)
Validation day: 4
Training cells: 29129
Validation cells: 8485
Mean baseline: 0.7909707422500121
Fitting fold-specific SVD...
Explained variance: 0.13588550686836243
SVD time: 0.4902814388275146 minutes
RNA-only score: 0.8757187004831929
Final feature dimension: 135
RNA + cell-type score: 0.8759327650398451
Cell-type gain: 0.0002140645566521826
Saved: /kaggle/working/cite_strict_results/cite_strict_fold_13176_summary.csv
Saved: /kaggle/working/cite_strict_results/cite_strict_fold_13176_cell_scores.csv

Strict CITE validation
Held-out donor: 31800
Training days: (2, 3)
Validation day: 4
Training cells: 28189
Validation cells: 10149
Mean baseline: 0.7579429419120671
Fitting fold-specific SVD...
Explained variance: 0.1451677680015564
SVD time: 0.4828409830729167 minutes
RNA-only score: 0.8554969987261597
Final feature dimension: 135
RNA + cell-type score: 0.8560322022903629
Cell-type gain: 0.0005352035642032238
Saved: /ka

In [27]:
fold_32606_path = Path(
    "/kaggle/working/"
    "cite_strict_fold_32606_summary.csv"
)

if not fold_32606_path.is_file():
    raise FileNotFoundError(
        "没有找到 32606 fold summary，"
        "请先运行第一步的保存代码。"
    )

fold_32606 = pd.read_csv(
    fold_32606_path
)

all_fold_summaries = pd.concat(
    [fold_32606] + new_fold_summaries,
    ignore_index=True
)

all_fold_summaries[
    "held_out_donor"
] = (
    all_fold_summaries[
        "held_out_donor"
    ]
    .astype(str)
)

all_fold_summaries = (
    all_fold_summaries
    .drop_duplicates(
        subset=["held_out_donor"],
        keep="last"
    )
    .sort_values("held_out_donor")
    .reset_index(drop=True)
)

display(all_fold_summaries)

,task,validation,held_out_donor,train_days,valid_day,n_train,n_valid,mean_baseline,model_score,absolute_improvement,svd_components,explained_variance,ridge_alpha,use_cell_type,rna_only_score,rna_celltype_score,celltype_gain,improvement_over_mean,svd_time_minutes
0,CITE,strict_donor_time,13176,"2,3",4,29129,8485,0.790971,NaN,NaN,128,0.135886,1000.0,NaN,0.875719,0.875933,0.000214,0.084962,0.490281
1,CITE,strict_donor_time,31800,"2,3",4,28189,10149,0.757943,NaN,NaN,128,0.145168,1000.0,NaN,0.855497,0.856032,0.000535,0.098089,0.482841
2,CITE,strict_donor_time,32606,"2,3",4,28368,9511,0.776165,0.879102,0.102938,128,0.143644,1000.0,True,0.878944,0.879102,0.000159,NaN,NaN


In [28]:
aggregate_summary = pd.DataFrame(
    [
        {
            "n_folds": len(
                all_fold_summaries
            ),
            "mean_baseline_mean": (
                all_fold_summaries[
                    "mean_baseline"
                ].mean()
            ),
            "mean_baseline_std": (
                all_fold_summaries[
                    "mean_baseline"
                ].std(ddof=1)
            ),
            "rna_only_mean": (
                all_fold_summaries[
                    "rna_only_score"
                ].mean()
            ),
            "rna_only_std": (
                all_fold_summaries[
                    "rna_only_score"
                ].std(ddof=1)
            ),
            "rna_celltype_mean": (
                all_fold_summaries[
                    "rna_celltype_score"
                ].mean()
            ),
            "rna_celltype_std": (
                all_fold_summaries[
                    "rna_celltype_score"
                ].std(ddof=1)
            ),
            "celltype_gain_mean": (
                all_fold_summaries[
                    "celltype_gain"
                ].mean()
            ),
            "celltype_gain_std": (
                all_fold_summaries[
                    "celltype_gain"
                ].std(ddof=1)
            ),
        }
    ]
)

display(aggregate_summary)

,n_folds,mean_baseline_mean,mean_baseline_std,rna_only_mean,rna_only_std,rna_celltype_mean,rna_celltype_std,celltype_gain_mean,celltype_gain_std
0,3,0.775026,0.016543,0.870053,0.012709,0.870356,0.012505,0.000303,0.000203


In [29]:
ALL_FOLDS_PATH = (
    STRICT_OUTPUT_DIR
    / "cite_strict_all_folds.csv"
)

AGGREGATE_PATH = (
    STRICT_OUTPUT_DIR
    / "cite_strict_aggregate.csv"
)

all_fold_summaries.to_csv(
    ALL_FOLDS_PATH,
    index=False
)

aggregate_summary.to_csv(
    AGGREGATE_PATH,
    index=False
)

print("Saved:")
print(ALL_FOLDS_PATH)
print(AGGREGATE_PATH)

Saved:
/kaggle/working/cite_strict_results/cite_strict_all_folds.csv
/kaggle/working/cite_strict_results/cite_strict_aggregate.csv


In [30]:
display(all_fold_summaries)
display(aggregate_summary)

,task,validation,held_out_donor,train_days,valid_day,n_train,n_valid,mean_baseline,model_score,absolute_improvement,svd_components,explained_variance,ridge_alpha,use_cell_type,rna_only_score,rna_celltype_score,celltype_gain,improvement_over_mean,svd_time_minutes
0,CITE,strict_donor_time,13176,"2,3",4,29129,8485,0.790971,NaN,NaN,128,0.135886,1000.0,NaN,0.875719,0.875933,0.000214,0.084962,0.490281
1,CITE,strict_donor_time,31800,"2,3",4,28189,10149,0.757943,NaN,NaN,128,0.145168,1000.0,NaN,0.855497,0.856032,0.000535,0.098089,0.482841
2,CITE,strict_donor_time,32606,"2,3",4,28368,9511,0.776165,0.879102,0.102938,128,0.143644,1000.0,True,0.878944,0.879102,0.000159,NaN,NaN


,n_folds,mean_baseline_mean,mean_baseline_std,rna_only_mean,rna_only_std,rna_celltype_mean,rna_celltype_std,celltype_gain_mean,celltype_gain_std
0,3,0.775026,0.016543,0.870053,0.012709,0.870356,0.012505,0.000303,0.000203


In [31]:
clean_fold_results = (
    all_fold_summaries.copy()
)

# 统一最终模型分数
clean_fold_results[
    "final_model_score"
] = clean_fold_results[
    "rna_celltype_score"
].fillna(
    clean_fold_results[
        "model_score"
    ]
)

# 统一相对 mean baseline 的绝对提升
clean_fold_results[
    "improvement_over_mean"
] = clean_fold_results[
    "improvement_over_mean"
].fillna(
    clean_fold_results[
        "absolute_improvement"
    ]
)

# 三个 fold 都使用了 cell type
clean_fold_results[
    "use_cell_type"
] = True

final_columns = [
    "held_out_donor",
    "train_days",
    "valid_day",
    "n_train",
    "n_valid",
    "mean_baseline",
    "rna_only_score",
    "final_model_score",
    "celltype_gain",
    "improvement_over_mean",
    "svd_components",
    "explained_variance",
    "ridge_alpha",
]

clean_fold_results = (
    clean_fold_results[final_columns]
    .sort_values("held_out_donor")
    .reset_index(drop=True)
)

display(clean_fold_results)

,held_out_donor,train_days,valid_day,n_train,n_valid,mean_baseline,rna_only_score,final_model_score,celltype_gain,improvement_over_mean,svd_components,explained_variance,ridge_alpha
0,13176,"2,3",4,29129,8485,0.790971,0.875719,0.875933,0.000214,0.084962,128,0.135886,1000.0
1,31800,"2,3",4,28189,10149,0.757943,0.855497,0.856032,0.000535,0.098089,128,0.145168,1000.0
2,32606,"2,3",4,28368,9511,0.776165,0.878944,0.879102,0.000159,0.102938,128,0.143644,1000.0


In [32]:
weights = clean_fold_results[
    "n_valid"
].to_numpy()

weighted_results = {
    "weighted_mean_baseline": np.average(
        clean_fold_results[
            "mean_baseline"
        ],
        weights=weights
    ),
    "weighted_rna_only": np.average(
        clean_fold_results[
            "rna_only_score"
        ],
        weights=weights
    ),
    "weighted_final_model": np.average(
        clean_fold_results[
            "final_model_score"
        ],
        weights=weights
    ),
    "weighted_celltype_gain": np.average(
        clean_fold_results[
            "celltype_gain"
        ],
        weights=weights
    ),
}

weighted_results

{'weighted_mean_baseline': np.float64(0.774057571473442),
 'weighted_rna_only': np.float64(0.8695166345821955),
 'weighted_final_model': np.float64(0.8698277801157721),
 'weighted_celltype_gain': np.float64(0.00031114553357666064)}

In [33]:
final_aggregate = pd.DataFrame(
    [
        {
            "validation": "strict donor + time",
            "n_folds": len(
                clean_fold_results
            ),
            "total_validation_cells": int(
                clean_fold_results[
                    "n_valid"
                ].sum()
            ),
            "mean_baseline_mean": (
                clean_fold_results[
                    "mean_baseline"
                ].mean()
            ),
            "mean_baseline_std": (
                clean_fold_results[
                    "mean_baseline"
                ].std(ddof=1)
            ),
            "rna_only_mean": (
                clean_fold_results[
                    "rna_only_score"
                ].mean()
            ),
            "rna_only_std": (
                clean_fold_results[
                    "rna_only_score"
                ].std(ddof=1)
            ),
            "final_model_mean": (
                clean_fold_results[
                    "final_model_score"
                ].mean()
            ),
            "final_model_std": (
                clean_fold_results[
                    "final_model_score"
                ].std(ddof=1)
            ),
            "weighted_final_model": (
                weighted_results[
                    "weighted_final_model"
                ]
            ),
            "celltype_gain_mean": (
                clean_fold_results[
                    "celltype_gain"
                ].mean()
            ),
            "celltype_gain_std": (
                clean_fold_results[
                    "celltype_gain"
                ].std(ddof=1)
            ),
            "improvement_over_mean": (
                clean_fold_results[
                    "improvement_over_mean"
                ].mean()
            ),
        }
    ]
)

display(final_aggregate)

,validation,n_folds,total_validation_cells,mean_baseline_mean,mean_baseline_std,rna_only_mean,rna_only_std,final_model_mean,final_model_std,weighted_final_model,celltype_gain_mean,celltype_gain_std,improvement_over_mean
0,strict donor + time,3,28145,0.775026,0.016543,0.870053,0.012709,0.870356,0.012505,0.869828,0.000303,0.000203,0.09533


In [34]:
FINAL_FOLDS_PATH = (
    STRICT_OUTPUT_DIR
    / "cite_strict_final_folds.csv"
)

FINAL_AGGREGATE_PATH = (
    STRICT_OUTPUT_DIR
    / "cite_strict_final_aggregate.csv"
)

clean_fold_results.to_csv(
    FINAL_FOLDS_PATH,
    index=False
)

final_aggregate.to_csv(
    FINAL_AGGREGATE_PATH,
    index=False
)

print("Saved:")
print(FINAL_FOLDS_PATH)
print(FINAL_AGGREGATE_PATH)

Saved:
/kaggle/working/cite_strict_results/cite_strict_final_folds.csv
/kaggle/working/cite_strict_results/cite_strict_final_aggregate.csv
